In [1]:
import pandas as pd
import numpy as np
import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import yfinance as yf
import sys
sys.path.append('../utils')
from indicators import *

# ignore all warnings
import warnings
warnings.filterwarnings('ignore')

Stock Market Simulation

-> generate multiple simulation using brownian motion graph and pick the one that's closest

In [2]:
# btc_usd=yf.download(tickers='BTC-USD', interval='1m', period='7d')
# btc_usd.reset_index(inplace=True)

In [3]:
# btc_usd['Datetime'] = btc_usd['Datetime'].dt.tz_convert('America/Los_Angeles')

In [4]:
# fin_df = pd.read_csv('btc_usd_1min.csv', usecols= ['Datetime', 'Adj Close'])

# fin_df.rename(columns={'Adj Close': 'price'}, inplace=True)
# fin_df['sma_5'] = get_sma(fin_df['price'], 5)
# fin_df['sma_20'] = get_sma(fin_df['price'], window=20)
# fin_df['rsi'] = get_rsi(fin_df['price'])
# fin_df['macd'], fin_df['signal'], fin_df['macd_hist'] = get_macd_signal(fin_df['price'])

fin_df.dropna(inplace=True)

init_price = fin_df['price'].iloc[0]
fin_df['price'] = fin_df['price'] / init_price

old_df = fin_df[fin_df['Datetime'] < '2024-02-25']
new_df = fin_df[fin_df['Datetime'] >= '2024-02-25']

In [5]:
def create_seq_features(df, columns, n_steps):
    df = df.copy()

    for column in columns:
        for i in range(1, n_steps+1):
            df[f'{column}(t-{i})'] = df[column].shift(i)
    df.dropna(inplace=True)
    return df

columns_to_sequence = ['price']#, 'sma_5', 'sma_20', 'rsi'] # , 'macd', 'signal', 'macd_hist']
old_df = create_seq_features(old_df, columns_to_sequence, 30)
new_df = create_seq_features(new_df, columns_to_sequence, 30)

## Train and Test Split

In [6]:
features = old_df.drop(columns=columns_to_sequence + ['Datetime']) # Using only past data
targets = old_df['price']

In [7]:
# from sklearn.preprocessing import MinMaxScaler
# scaler = MinMaxScaler(feature_range=(0,1))
N = len(features)
training_idx = int(N * 0.9)

X_train, y_train = features[:training_idx].values, targets[:training_idx].values
X_test, y_test = features[training_idx:].values, targets[training_idx:].values

# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)


X_train = torch.from_numpy(X_train).to(torch.float32).reshape(-1, len(features.columns), 1)
y_train = torch.from_numpy(y_train).to(torch.float32).reshape(-1,1)
X_test = torch.from_numpy(X_test).to(torch.float32).reshape(-1, len(features.columns),1)
y_test = torch.from_numpy(y_test).to(torch.float32).reshape(-1,1)

### Training

In [8]:
from torch.utils.data import Dataset, DataLoader
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, i):
        return self.X[i], self.y[i]

training = TimeSeriesDataset(X_train, y_train)
test = TimeSeriesDataset(X_test, y_test)

BATCH_SIZE = 32
train_loader = DataLoader(training, BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test, BATCH_SIZE, shuffle=False)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [10]:
class QLSTM(nn.Module):
  def __init__(self, input_size, hidden_size, num_layers, output_size=None):
    super(QLSTM, self).__init__()
    # If output_size is not given, use input_size
    if output_size is None:
      output_size = input_size
    self.input_size = input_size      # the size of the input
    self.hidden_size = hidden_size    # the size of the hidden state
    self.num_layers = num_layers      # the size of lstm layers
    self.output_size = output_size    # the size of the output (if different)
    
    self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
    self.dropout1 = nn.Dropout(0.2)
    self.lin_layer2 = nn.Linear(hidden_size, output_size)
    # self.dropout = nn.Dropout(p=0.2)

  def forward(self, x):
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
    # h = nn.init.kaiming_uniform_(h0)
    c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(device)
    # c = nn.init.kaiming_uniform_(c0)
    # nn.init.kaiming_uniform_(h0, a=np.sqrt(5))  # Example usage
    # nn.init.kaiming_uniform_(c0, a=np.sqrt(5))
    output, _ = self.lstm(x, (h0,c0))
    output = self.dropout1(output)
    output = output[:, -1, :]
    # output = self.lin_layer(output)
    output = self.lin_layer2(output)
    
    return output

In [11]:
# Train the model
def train(model, train_loader, epochs, optimizer, criterion, patience=10, min_delta=1e-8):
  model.train()
  model.to(device)
  best_loss = float('inf')
  no_improve_epoch = 0
  losses = []
  for epoch in range(epochs):
    total_loss = 0.0
    for _, batch in enumerate(train_loader):
      x_batch, y_batch = batch[0].to(device), batch[1].to(device)
      optimizer.zero_grad()

      output  = model(x_batch)
      loss = criterion(output, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
    epoch_loss = total_loss / len(train_loader)
    losses.append(epoch_loss)
    if (epoch+1)%10 == 0:
      print(f'Epoch: {epoch+1}, Loss: {epoch_loss}')

    if best_loss - epoch_loss > min_delta:
        best_loss = epoch_loss
        no_improve_epoch = 0
    else:
        no_improve_epoch += 1
    if no_improve_epoch >= patience:
        print(f'No improvement in loss for {patience} consecutive epochs. Stopping training.')
        break
  return model, losses

qlstm = QLSTM(input_size=X_train.shape[-1], hidden_size=512, num_layers=2, output_size=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(qlstm.parameters(), lr=0.001)


In [12]:
# It's ok to modify this cell.
# BATCH_SIZE = 50
N_EPOCHS = 10000

In [13]:

train(qlstm, train_loader, N_EPOCHS, optimizer, criterion)

Epoch: 10, Loss: 0.000613265881043395
Epoch: 20, Loss: 0.0005223398713224383
Epoch: 30, Loss: 0.00038712249933515753
Epoch: 40, Loss: 0.000351195513077084
Epoch: 50, Loss: 0.00020828405678529585
Epoch: 60, Loss: 0.0001677335136993515
Epoch: 70, Loss: 0.00012654731304232532
Epoch: 80, Loss: 9.932640106397208e-05
Epoch: 90, Loss: 9.297235623756866e-05
Epoch: 100, Loss: 6.86255543437826e-05
Epoch: 110, Loss: 5.9248713180636054e-05
Epoch: 120, Loss: 4.78911648804989e-05
Epoch: 130, Loss: 4.2078840531183615e-05
Epoch: 140, Loss: 5.0205860249223606e-05
No improvement in loss for 10 consecutive epochs. Stopping training.


(QLSTM(
   (lstm): LSTM(1, 512, num_layers=2, batch_first=True)
   (dropout1): Dropout(p=0.2, inplace=False)
   (lin_layer2): Linear(in_features=512, out_features=1, bias=True)
 ),
 [0.09805192603981348,
  0.0012311832008408269,
  0.0009992812269533182,
  0.000888740002042859,
  0.0007358986934079498,
  0.0007444943261553386,
  0.0007148155372736877,
  0.0006478369753969916,
  0.0006084007118887038,
  0.000613265881043395,
  0.0006336232579901422,
  0.000553761481112793,
  0.0005830840962568509,
  0.0005622430467579386,
  0.0005293650543086254,
  0.0005060195969276296,
  0.0005120206595401314,
  0.0005223838241157208,
  0.000533796124985566,
  0.0005223398713224383,
  0.000504396850396271,
  0.0004992787185976251,
  0.00047767548396502837,
  0.0004763717031170452,
  0.00047425555123246243,
  0.00044041820432988904,
  0.0004679385444768214,
  0.000426750565393119,
  0.0003899842021592821,
  0.00038712249933515753,
  0.00035738970081393537,
  0.0003963425604208007,
  0.000378623223302203

In [166]:
def test(model, test_loader, criterion):
    model.eval()  # Switch to evaluation mode
    total_loss = 0.0
    correct_predictions = 0
    total_predictions = 0

    with torch.no_grad():  # No need to track gradients for testing
        for _, batch in enumerate(test_loader):
            x_batch, y_batch = batch[0].to(device), batch[1].to(device)

            output = model(x_batch)
            loss = criterion(output, y_batch)
            total_loss += loss.item()
            # total_predictions += y_batch.size(0)
            # correct_predictions += (predicted == y_batch).sum().item()

    average_loss = total_loss / len(test_loader)
    accuracy = correct_predictions / total_predictions * 100

    print(f'Test Loss: {average_loss:.4f}, Accuracy: {accuracy:.2f}%')

    return average_loss, accuracy

In [14]:
y_pred_train = qlstm(X_train.to(device))

In [15]:
a = y_pred_train.detach().cpu()